# 1. Introdução
# Sistema Inteligente de Análise de Sintomas e Classificação de Risco

Este notebook demonstra um pipeline de **NLP e Machine Learning** em duas partes:

- **Parte 1:** extração de sintomas em frases e associação simulada a condições via mapa de conhecimento;
- **Parte 2:** classificação de frases em `baixo_risco` ou `alto_risco` com TF-IDF + Regressão Logística.

# 2. Objetivos

1. Normalizar textos de relatos simulados;
2. Identificar sintomas com um mapa de conhecimento e pontuar condições associadas;
3. Treinar e avaliar um classificador de risco;
4. Gerar métricas, gráficos e testar o modelo com frases novas.

# 3.finalidade acadêmica

> **Este projeto possui finalidade exclusivamente acadêmica e demonstrativa. Os dados, associações, classificações e resultados são simulados e não devem ser utilizados para diagnóstico, triagem ou tomada de decisão médica real.**

# 4. Arquitetura do projeto

```text
Dados -> Pré-processamento -> ┬-> Extração -> Mapa -> Associação ──┐
                              └-> TF-IDF -> Machine Learning -> Classificação -> Avaliação
```

A lógica reutilizável fica em `src/`; o notebook é a camada de execução, demonstração e visualização.

# 5. Importação das bibliotecas

Importamos bibliotecas de apoio e os módulos do projeto em `src/`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Garante que a raiz do projeto esteja no path para importar `src`
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.preprocessing import normalizar_texto, remover_acentos, limpar_texto
from src.mapa_conhecimento import carregar_mapa, normalizar_mapa, buscar_associacoes
from src.extracao_sintomas import carregar_frases, extrair_sintomas, processar_frases, salvar_resultados
from src.classificador_risco import (carregar_base_risco, preparar_base, dividir_treino_teste,
                                     treinar_modelo, prever)
from src.avaliacao import (avaliar_modelo, salvar_metricas, plotar_distribuicao_classes,
                           plotar_matriz_confusao, plotar_metricas)

# 6. Configuração dos caminhos

Todos os caminhos são relativos à raiz do projeto.

In [2]:
RAW = RAIZ / "data" / "raw"
PROCESSED = RAIZ / "data" / "processed"
RESULTS = RAIZ / "results"
FIGURAS = RESULTS / "figuras"
for pasta in (PROCESSED, RESULTS, FIGURAS):
    pasta.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ.name)

Raiz do projeto: sistema_analise_sintomas


# 7. Carregamento dos dados

Carregamos as 10 frases, o mapa de conhecimento e a base de risco (todos simulados).

In [3]:
frases = carregar_frases(RAW / "sintomas.txt")
mapa = carregar_mapa(RAW / "mapa_conhecimento.csv")
base = carregar_base_risco(RAW / "base_risco.csv")
print(f"Frases: {len(frases)} | Linhas do mapa: {len(mapa)} | Exemplos de risco: {len(base)}")
for i, f in enumerate(frases, 1):
    print(f"{i}. {f}")

Frases: 10 | Linhas do mapa: 38 | Exemplos de risco: 63
1. Há dois dias estou com uma dor no peito que piora quando faço esforço físico e me impede de subir escadas.
2. Sinto cansaço constante há uma semana, mesmo depois de descansar bem à noite, e isso atrapalha meu trabalho.
3. Estou com dificuldade para respirar desde ontem e fico muito cansado ao caminhar até o ponto de ônibus.
4. Desde a manhã de hoje tenho uma dor de cabeça forte e sensibilidade à luz, o que me obrigou a ficar no escuro.
5. Há três dias venho tendo tosse seca e febre alta à noite, e quase não consigo dormir direito.
6. Sinto tontura intensa quando levanto da cama e às vezes minha visão fica turva por alguns segundos.
7. Estou com náusea e dor no estômago depois das refeições há cerca de cinco dias, o que reduziu meu apetite.
8. Notei dor nas articulações e dor muscular no corpo todo desde o fim de semana, principalmente ao acordar.
9. Há duas semanas sinto palpitações e coração acelerado em momentos de repouso, o

# 8. Análise exploratória

Verificamos valores ausentes, duplicatas e o tamanho médio das frases.

In [4]:
print("Ausentes no mapa:", int(mapa.isna().sum().sum()), "| Ausentes na base:", int(base.isna().sum().sum()))
print("Duplicatas no mapa:", int(mapa.duplicated().sum()), "| Duplicatas na base:", int(base.duplicated('frase').sum()))
print("Média de palavras por frase:", round(base['frase'].str.split().str.len().mean(), 1))
mapa.head()

Ausentes no mapa: 0 | Ausentes na base: 0
Duplicatas no mapa: 0 | Duplicatas na base: 0
Média de palavras por frase: 11.9


             sintoma_1                  sintoma_2         condicao_associada
0         dor no peito            aperto no peito  Condição Cardiovascular A
1    cansaço constante                     fadiga  Condição Cardiovascular B
2          falta de ar  dificuldade para respirar    Condição Respiratória A
3  dor de cabeça forte        sensibilidade à luz     Condição Neurológica A
4           tosse seca                 febre alta    Condição Respiratória B

# 9. Pré-processamento

Convertemos para minúsculas, removemos acentos e pontuação e normalizamos espaços.

In [5]:
print(normalizar_texto("Dificuldade Para Respirar!"))
print(remover_acentos("cansaço"))
print(limpar_texto("dor!!   no   peito..."))
print(normalizar_texto(frases[0]))

dificuldade para respirar
cansaco
dor no peito
ha dois dias estou com uma dor no peito que piora quando faco esforco fisico e me impede de subir escadas


# 10. Parte 1 — Extração de sintomas

Fluxo: frase -> normalização -> busca no mapa -> condições -> pontuação -> principal associação (simulada).

# 11. Mapa de conhecimento

Normalizamos as expressões do mapa para que variações de acento/caixa sejam reconhecidas.

In [6]:
mapa_norm = normalizar_mapa(mapa)
print(buscar_associacoes(mapa_norm, "Dor no peito")[["sintoma_1", "sintoma_2", "condicao_associada"]])

       sintoma_1          sintoma_2         condicao_associada
0   dor no peito    aperto no peito  Condição Cardiovascular A
11  dor no peito        falta de ar  Condição Cardiovascular A
12  dor no peito  coração acelerado  Condição Cardiovascular C


# 12. Extração

Aplicamos a extração a uma frase de exemplo e depois a todas.

In [7]:
extrair_sintomas(frases[0], mapa_norm)

{'frase_original': 'Há dois dias estou com uma dor no peito que piora quando faço esforço físico e me impede de subir escadas.', 'frase_normalizada': 'ha dois dias estou com uma dor no peito que piora quando faco esforco fisico e me impede de subir escadas', 'sintomas_identificados': 'dor no peito', 'condicoes_encontradas': 'Condição Cardiovascular A (1); Condição Cardiovascular C (1)', 'condicao_principal': 'Condição Cardiovascular A | Condição Cardiovascular C', 'pontuacao': 1}

# 13. Pontuação

Cada sintoma encontrado soma **+1** para cada condição associada. A maior pontuação define a `condicao_principal`; em **empate**, todas as condições empatadas são exibidas (sem escolha arbitrária).

In [8]:
resultados = processar_frases(frases, mapa_norm)
resultados[["id", "sintomas_identificados", "condicoes_encontradas", "condicao_principal", "pontuacao"]]

   id  ... pontuacao
0   1  ...         1
1   2  ...         1
2   3  ...         1
3   4  ...         2
4   5  ...         2
5   6  ...         1
6   7  ...         2
7   8  ...         2
8   9  ...         2
9  10  ...         3

[10 rows x 5 columns]

# 14. Resultados

Salvamos `data/processed/resultados_processamento.csv` e `results/resultados_extracao.csv`.

In [9]:
resumo = salvar_resultados(resultados, PROCESSED / "resultados_processamento.csv",
                           RESULTS / "resultados_extracao.csv")
resumo

                                               frase  ... pontuacao
0  Há dois dias estou com uma dor no peito que pi...  ...         1
1  Sinto cansaço constante há uma semana, mesmo d...  ...         1
2  Estou com dificuldade para respirar desde onte...  ...         1
3  Desde a manhã de hoje tenho uma dor de cabeça ...  ...         2
4  Há três dias venho tendo tosse seca e febre al...  ...         2
5  Sinto tontura intensa quando levanto da cama e...  ...         1
6  Estou com náusea e dor no estômago depois das ...  ...         2
7  Notei dor nas articulações e dor muscular no c...  ...         2
8  Há duas semanas sinto palpitações e coração ac...  ...         2
9  Desde ontem estou com garganta inflamada e dor...  ...         3

[10 rows x 5 columns]

# 15. Parte 2 — Classificação de risco

Agora treinamos um modelo que classifica frases como `baixo_risco` ou `alto_risco` (rótulos simulados).

# 16. Preparação da base

Reutilizamos o **mesmo** módulo de pré-processamento da Parte 1.

In [10]:
base = preparar_base(base)
base[["frase", "frase_normalizada", "situacao"]].head()

                                               frase  ...     situacao
0  Estou com um pequeno desconforto muscular depo...  ...  baixo_risco
1  Sinto uma leve dor de cabeça no fim da tarde q...  ...  baixo_risco
2  Tive um resfriado simples com coriza e espirro...  ...  baixo_risco
3  Acordei com o pescoço um pouco rígido por ter ...  ...  baixo_risco
4  Estou com um cansaço leve depois de uma semana...  ...  baixo_risco

[5 rows x 3 columns]

# 17. Análise das classes

Verificamos o equilíbrio entre as classes e salvamos o Gráfico 1.

In [11]:
print(base["situacao"].value_counts())
plotar_distribuicao_classes(base, FIGURAS / "distribuicao_classes.png");

situacao
baixo_risco    32
alto_risco     31
Name: count, dtype: int64


# 18. Divisão treino/teste

Divisão de 80/20, estratificada, com `random_state=42`.

In [12]:
X_treino, X_teste, y_treino, y_teste = dividir_treino_teste(base)
print("Treino:", len(X_treino), "| Teste:", len(X_teste))

Treino: 50 | Teste: 13


# 19. TF-IDF

- **Por que transformar texto em números?** Algoritmos de ML operam sobre números, não sobre palavras.
- **TF (Term Frequency):** frequência de um termo em uma frase.
- **IDF (Inverse Document Frequency):** reduz o peso de termos muito comuns na base e destaca os mais discriminativos (`log(N / df)`, com suavização no scikit-learn).
- **TF-IDF = TF x IDF:** pondera termos frequentes na frase, mas raros no conjunto.
- **Unigramas:** palavras isoladas (`dor`, `peito`). **Bigramas:** pares consecutivos (`dor no`, `no peito`), que preservam parte do contexto.

Usamos `TfidfVectorizer(lowercase=True, ngram_range=(1, 2))`.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
vetorizador_demo = TfidfVectorizer(lowercase=True, ngram_range=(1, 2)).fit(X_treino)
print("Tamanho do vocabulário (unigramas + bigramas):", len(vetorizador_demo.vocabulary_))
print("Exemplos:", list(vetorizador_demo.vocabulary_)[:8])

Tamanho do vocabulário (unigramas + bigramas): 674
Exemplos: ['apareceu', 'uma', 'coceira', 'discreta', 'no', 'braco', 'depois', 'de']


# 20. Treinamento

O modelo principal é o pipeline **TF-IDF -> LogisticRegression**.

In [14]:
modelo = treinar_modelo(X_treino, y_treino)
modelo

Pipeline(steps=[('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
                ('modelo', LogisticRegression(max_iter=1000, random_state=42))])

# 21. Previsões

Prevemos as classes do conjunto de teste.

In [15]:
y_previsto = modelo.predict(X_teste)
pd.DataFrame({"frase": X_teste.values, "real": y_teste.values, "previsto": y_previsto}).head(10)

                                               frase         real     previsto
0  meu rosto e a lingua estao inchando e tenho di...   alto_risco   alto_risco
1  notei uma mancha pequena na pele que nao coca ...  baixo_risco   alto_risco
2  senti um leve mal estar depois de tomar sol po...  baixo_risco  baixo_risco
3  estou com dificuldade grave para respirar e me...   alto_risco   alto_risco
4  sinto uma leve tontura apenas quando fico muit...  baixo_risco  baixo_risco
5  sinto uma dor muito forte e repentina na cabec...   alto_risco   alto_risco
6  sinto uma dor leve nas costas quando fico sent...  baixo_risco  baixo_risco
7  tive uma convulsao ha pouco e ainda estou confuso   alto_risco  baixo_risco
8  sinto o corpo um pouco cansado mas consigo tra...  baixo_risco  baixo_risco
9  estou com um desconforto no estomago depois de...  baixo_risco  baixo_risco

# 22. Avaliação

Calculamos accuracy, precision, recall e F1-score (média ponderada) e salvamos `results/metricas_modelo.csv`.

In [16]:
metricas, matriz, relatorio = avaliar_modelo("LogisticRegression", y_teste, y_previsto)
salvar_metricas(metricas, RESULTS / "metricas_modelo.csv")
print(relatorio)
plotar_metricas(metricas, FIGURAS / "metricas_modelo.png")
metricas.round(4)

              precision    recall  f1-score   support

 baixo_risco       0.86      0.86      0.86         7
  alto_risco       0.83      0.83      0.83         6

    accuracy                           0.85        13
   macro avg       0.85      0.85      0.85        13
weighted avg       0.85      0.85      0.85        13



               modelo  accuracy  precision  recall  f1_score
0  LogisticRegression    0.8462     0.8462  0.8462    0.8462

# 23. Matriz de confusão

Linhas = classe real; colunas = classe prevista.

In [17]:
plotar_matriz_confusao(matriz, FIGURAS / "matriz_confusao.png")
print(matriz)

[[6 1]
 [1 5]]


# 24. Testes com frases novas

Frases inéditas, que não estão na base de treinamento.

In [18]:
novas_frases = [
    "Estou sentindo apenas um pequeno desconforto muscular.",
    "Tenho uma leve coceira no braço depois de usar um creme novo.",
    "Estou com um resfriado fraco e o nariz escorrendo um pouco.",
    "Senti um cansaço leve depois de acordar cedo hoje.",
    "Meu joelho incomoda um pouco depois da corrida.",
    "Estou com uma dor muito forte no peito e muita falta de ar.",
    "Desmaiei de repente e não consigo mover o lado esquerdo do corpo.",
    "Tenho febre altíssima com rigidez no pescoço e confusão mental.",
    "Estou vomitando sangue e sentindo uma dor abdominal insuportável.",
    "Meus lábios estão roxos e mal consigo respirar.",
]
assert not set(novas_frases) & set(base["frase"]), "Há frases repetidas da base!"
teste = prever(modelo, novas_frases)
for _, linha in teste.iterrows():
    print(f"Frase: {linha['frase']}\n  -> {linha['classificacao_prevista']}\n")

Frase: Estou sentindo apenas um pequeno desconforto muscular.
  -> baixo_risco

Frase: Tenho uma leve coceira no braço depois de usar um creme novo.
  -> baixo_risco

Frase: Estou com um resfriado fraco e o nariz escorrendo um pouco.
  -> baixo_risco

Frase: Senti um cansaço leve depois de acordar cedo hoje.
  -> baixo_risco

Frase: Meu joelho incomoda um pouco depois da corrida.
  -> baixo_risco

Frase: Estou com uma dor muito forte no peito e muita falta de ar.
  -> alto_risco

Frase: Desmaiei de repente e não consigo mover o lado esquerdo do corpo.
  -> alto_risco

Frase: Tenho febre altíssima com rigidez no pescoço e confusão mental.
  -> alto_risco

Frase: Estou vomitando sangue e sentindo uma dor abdominal insuportável.
  -> alto_risco

Frase: Meus lábios estão roxos e mal consigo respirar.
  -> alto_risco



# 25. Análise crítica

- O modelo aprende principalmente **palavras indicativas** (ex.: "forte", "intensa", "desmaio" vs. "leve", "pequeno").
- Frases com negação, ironia ou vocabulário ausente no treinamento podem ser classificadas incorretamente.
- As métricas dependem de um conjunto de teste **muito pequeno**, portanto são instáveis.